In [38]:
!pip install ultralytics supervision opencv-python numpy matplotlib tqdm


In [39]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

import supervision as sv

In [40]:
print("OpenCV :", cv2.__version__)
print("Supervision :", sv.__version__)

OpenCV : 4.10.0
Supervision : 0.29.1


In [41]:
VIDEO_PATH = "people-walking.mp4"          
OUTPUT_VIDEO_PATH = "output/people_flow_output.mp4"
OUTPUT_HEATMAP_PATH = "output/final_heatmap.png"
OUTPUT_OVERLAY_PATH = "output/heatmap_overlay.png"

model = YOLO("yolo11n.pt")   
PERSON_CLASS_ID = 0          

In [42]:
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
width, height = video_info.width, video_info.height
fps = video_info.fps
total_frames = video_info.total_frames

print(width, height, fps, total_frames)

1920 1080 25.0 341


In [43]:
tracker = sv.ByteTrack()
bounding_box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

In [44]:
UPPER_LINE = 220
LOWER_LINE = 420

IN_COUNT = 0
OUT_COUNT = 0

history = {}
counted_in = set()
counted_out = set()

heatmap = np.zeros((height, width), dtype=np.float32)

In [45]:
def get_center(bbox):
    x1, y1, x2, y2 = bbox
    cx = int((x1 + x2) / 2)
    cy = int((y1 + y2) / 2)
    return cx, cy

def update_heatmap(heatmap, cx, cy, radius=18, intensity=1.0):
    cv2.circle(heatmap, (cx, cy), radius, intensity, -1)

def check_crossing(tid, cy):
    global IN_COUNT, OUT_COUNT
    if tid not in history:
        return
    prev_y = history[tid]

    if prev_y < UPPER_LINE <= cy and tid not in counted_in:
        IN_COUNT += 1
        counted_in.add(tid)

    if prev_y > LOWER_LINE >= cy and tid not in counted_out:
        OUT_COUNT += 1
        counted_out.add(tid)

In [46]:
import os
from tqdm import tqdm

cap = cv2.VideoCapture(VIDEO_PATH)
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

for _ in tqdm(range(total_frames), desc="Processing video"):
    ret, frame = cap.read()
    if not ret:
        break

    result = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    detections = detections[detections.class_id == PERSON_CLASS_ID]
    detections = tracker.update_with_detections(detections)

    labels = [f"ID {tid}" for tid in detections.tracker_id]

    for bbox, tid in zip(detections.xyxy, detections.tracker_id):
        cx, cy = get_center(bbox)
        update_heatmap(heatmap, cx, cy)
        check_crossing(tid, cy)
        history[tid] = cy

    annotated = bounding_box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

    cv2.line(annotated, (0, UPPER_LINE), (width, UPPER_LINE), (0, 255, 0), 2)
    cv2.putText(annotated, "UPPER (IN)", (10, UPPER_LINE - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.line(annotated, (0, LOWER_LINE), (width, LOWER_LINE), (0, 0, 255), 2)
    cv2.putText(annotated, "LOWER (OUT)", (10, LOWER_LINE - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.putText(annotated, f"IN: {IN_COUNT}", (width - 200, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(annotated, f"OUT: {OUT_COUNT}", (width - 200, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    writer.write(annotated)

cap.release()
writer.release()

print("IN:", IN_COUNT, " OUT:", OUT_COUNT)

os.startfile(os.path.abspath(OUTPUT_VIDEO_PATH))

Processing video: 100%|██████████| 341/341 [00:24<00:00, 14.09it/s]


IN: 7  OUT: 9


In [47]:
import os

heatmap_blur = cv2.GaussianBlur(heatmap, (0, 0), sigmaX=20, sigmaY=20)
heatmap_norm = cv2.normalize(heatmap_blur, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)

cv2.imwrite(OUTPUT_HEATMAP_PATH, heatmap_color)

cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

overlay = cv2.addWeighted(first_frame, 0.6, heatmap_color, 0.4, 0)
cv2.imwrite(OUTPUT_OVERLAY_PATH, overlay)

print("Heatmap saved:", OUTPUT_HEATMAP_PATH)
print("Overlay saved:", OUTPUT_OVERLAY_PATH)

os.startfile(os.path.abspath(OUTPUT_HEATMAP_PATH))
os.startfile(os.path.abspath(OUTPUT_OVERLAY_PATH))

Heatmap saved: output/final_heatmap.png
Overlay saved: output/heatmap_overlay.png
